# 03 — Sampler dynamics：Full-space、Fixed-cardinality、Effective Exchange 与 Reverse Annealing

## 当前定位

这一阶段的研究目标是：

$$
\boxed{\text{尽量保持问题 Hamiltonian 不变，只优化 sampler dynamics}}
$$

当前优先 full-space；hard fixed-cardinality 作为重要对照；reverse annealing 暂缓到后续。

## 1. Full-space 与 fixed-cardinality 双路线

### Full-space（优先）

$$
\Omega=\{0,1\}^{261}.
$$

保留：

$$
H_{\mathrm{count}}
=
A_{\mathrm{count}}
\left(\sum_e x_e-143\right)^2.
$$

允许 cardinality 改变。

只要保留 single-bit channel，整个 binary space 仍可达。

### Fixed-cardinality（对照）

$$
\Omega_{143}
=
\left\{x:\sum_e x_e=143\right\}.
$$

此时：

$$
H_{\mathrm{count}}\equiv0.
$$

可删掉 dense count square。

优势是搜索几何更贴近 Hamilton path；缺点是这不是 standard independent transverse-field driver 的完整状态空间。

## 2. Kernel 命名

### Exchange move size：$k_{\mathrm{ex}}$

$$
k_{\mathrm{ex}}\text{ 个 selected edges 关闭}
$$

同时：

$$
k_{\mathrm{ex}}\text{ 个 unselected edges 打开}.
$$

因此：

$$
\Delta|E_x|=0.
$$

### Reconnect

当前为 directed 2-switch：

$$
(a\to b,c\to d)
\leftrightarrow
(a\to d,c\to b).
$$

这是独立于 $k_{\mathrm{ex}}$ 的 4-bit topology move。

### Insertion / removal

当前测试了：

$$
\{a\to b\}
\leftrightarrow
\{a\to c,c\to b\}.
$$

它是一个 3-bit move：

$$
\Delta |E_x|=\pm1.
$$

它属于 full-space cardinality-changing kernel。

## 3. 为什么 reconnect 值得比随机 $k_{\mathrm{ex}}>1$ 更优先优化

随机 $k_{\mathrm{ex}}=2,3$ 的接受率明显低于 $k=1$，即使后期增加多次独立尝试，也没有稳定改善。

而 reconnect 有不同性质：

$$
d_v^{in},d_v^{out}
$$

完全保持。

因此在 degree 已经正确以后，它可以直接改变 cycle / component topology，而不重新破坏 degree。

我们已经有一个真实成功轨迹：

$$
1{,}326{,}008
\xrightarrow{\text{late reconnect}}
1{,}343{,}093.
$$

所以接下来更值得做：

- reconnect intensity；
- 3-edge reconnect；
- cycle-boundary reconnect；
- component-endpoint reconnect；

而不是继续单纯增加随机 $k_{\mathrm{ex}}=3$ 次数。

## 4. Dynamic kernel mixture

若每个子 kernel 本身满足：

$$
q_i(x\to x')=q_i(x'\to x),
$$

且 mixture 权重只依赖时间：

$$
p_i=p_i(t),
$$

则：

$$
q_t(x\to x')
=
\sum_i p_i(t)q_i(x\to x')
$$

仍然对称。

因此可以做 anneal-time / outer-iteration dependent schedule。

但如果：

$$
p_i=p_i(x),
$$

例如“degree conflict 小于 2 就增加 reconnect”，proposal 一般不再自动对称，需要显式 Metropolis-Hastings correction。

## 5. Multi-bit proposal 是否仍算标准 SQA？

关键不是“是否一次翻多个 bits”。

只要 full-space 不变，并且 sampler 仍针对同一个 Suzuki–Trotter effective distribution 做 detailed-balance MCMC，multi-bit / cluster proposal 可以只是采样加速。

因此：

$$
\boxed{\text{multi-bit MCMC}\neq\text{自动改变 quantum driver}}
$$

真正与标准 independent transverse field 不一致的是：

$$
\boxed{\text{把状态空间硬限制在 fixed Hamming weight sector}}
$$

因为：

$$
-\Gamma\sum_i\sigma_i^x
$$

会单独翻一个 bit，从而改变 Hamming weight。

## 6. Effective second-order exchange：为什么两条路线并非毫无关系

考虑 full-space 中强 count penalty：

$$
H_P
\supset
A_{\mathrm{count}}
\left(\sum_i x_i-K\right)^2
$$

以及标准 transverse field：

$$
H_D=-\Gamma\sum_i\sigma_i^x.
$$

在 $K$-edge 低能 sector 中，一个单 bit flip 会暂时离开该 sector：

$$
10\to00
$$

或：

$$
10\to11.
$$

第二次 flip 可回到：

$$
01.
$$

所以低能有效动力学中会出现 exchange-like：

$$
10\leftrightarrow01.
$$

量级预期类似：

$$
J_{\mathrm{eff}}
\sim
\frac{\Gamma^2}{\Delta_{\mathrm{count}}},
$$

但这目前只是物理图像，还没有做正式 Schrieffer–Wolff / degenerate perturbation derivation。

意义是：

- full-space：exchange 通过高能 virtual states 间接发生；
- fixed-cardinality sampler：直接把 exchange 当作一次 proposal。

这为两条路线提供了形式联系。

## 7. Full-space 第一轮实验带来的新问题

full-space + count penalty 后，sampler 很容易达到：

$$
R_{\deg}=0,
$$

但 selected edges 常停在：

$$
139\sim142
$$

而不是 143。

原因是少选边有利于消除 degree penalty；一旦成为 under-filled path cover，balanced exchange / reconnect 都满足：

$$
\Delta|E_x|=0
$$

无法补回缺边。

single-bit addition 又可能需要先制造高能 degree/count 中间态。

因此 full-space 的核心问题是：

$$
\boxed{\text{cardinality + degree 联合 barrier}}
$$

## 8. Insertion/removal 的第一版结果

测试 move：

$$
\{a\to b\}
\leftrightarrow
\{a\to c,c\to b\}.
$$

complex144 中有 134 个 static templates。

优点：

- proposal 可设计成 involution；
- 能改变 cardinality；
- 比随机 single addition 更结构化。

观察：

- eligibility 约 64%；
- conditional acceptance 仍非常低；
- $A_{\mathrm{count}}=32$ 与 64 都未稳定得到 143-edge Hamilton path。

因此下一步应从简单 1↔2 扩展成更能同时修结构的：

$$
k\leftrightarrow k+1
$$

局部 reroute。

## 9. Reverse annealing：记录想法，暂缓实验

当前 projected SQA 即使 warm-start 上一轮结果，仍是：

$$
\Gamma:\text{high}\to\text{low},
$$

因此是 warm-started forward SQA，不是 reverse annealing。

真正 reverse annealing 从已知 classical state 出发：

$$
\Gamma\approx0
\to
\Gamma_{\mathrm{peak}}
\to
0.
$$

它最适合：

1. 已有合法 Hamilton path，做 weighted refinement；
2. forward search 反复回到同一 basin；
3. 多个 feasible seed 的局部 refinement；
4. 研究 complex144 多-path basin 之间是否可跨越。

该方向暂缓，待 full-space forward sampler 更清楚以后再单独做。

## 10. 当前实验路线

### 优先

$$
\boxed{\text{Full-space + structured multi-bit kernels}}
$$

重点：

- single-bit 保证全空间可达；
- exchange 做近 fixed-cardinality motion；
- reconnect 做 topology change；
- insertion/removal 或 $k\leftrightarrow k+1$ 做 cardinality repair。

### 对照

$$
\boxed{\text{Hard fixed-cardinality}}
$$

用于量化“直接把 hard constraint 写进 sampler geometry”能获得多少性能收益。

### 暂缓

- reverse annealing；
- compression 作为默认前处理；
- 继续盲目增加随机 $k_{\mathrm{ex}}>1$。

## 11. 最新补充：单纯增加 reconnect 次数也不够

为确认“真正值得优化的是 reconnect”这一点，我们测试了一个最简单的 reconnect 强度实验。

前 12 个 outer iterations 仍使用 $k_{\mathrm{ex}}=1$ exchange 修 degree；后 6 个 iterations 中，每个 slice/sweep 在一次 exchange 之外额外执行：

- 1 次 reconnect；
- 2 次 reconnect；
- 4 次 reconnect。

每个 reconnect 都是独立的 symmetric Metropolis transition，不做 best-of-many 选择。

4 seeds/config 的结果中，三组都没有得到 Hamilton path；最低 degree conflict 都只能到 1 左右。增加 reconnect 次数把 native proposal 数从约 46 万提高到约 81 万，但没有带来明显突破。

因此当前结论不是“reconnect 没用”。此前已有一次 reconnect phase 把

$$
1{,}326{,}008
\to
1{,}343{,}093.
$$

真正说明的是：

$$
\boxed{\text{reconnect 的价值在于结构，而不是单纯多试几次同一个 2-switch。}}
$$

下一步更值得测试的是：

- 3-edge reconnect；
- cycle-boundary reconnect；
- component-endpoint reconnect；
- 在 degree 已经为 0 时专门生成能打断 cycle / 合并 component 的 reconnect。

这和随机增加 $k_{\mathrm{ex}}>1$ 次数得到的结论一致：**需要更有结构的联合 move，而不是更多无条件随机 move。**